# Tutorial 8: Train NicheTrans on human lymph node data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_human_lymph_node import Lymph_node

from utils.utils import *
from utils.utils_training_human_lymph_node import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_human_lymph_node.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.1, n_source=3000, workers=4, adata_path='/mnt/datadisk0/Processed_DATA/2024_nm_human_lymph_nodes/', cell_type_visualize=False, cell_type_visualization_dir=None, cell_type_visualization_dpi=150, max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = Lymph_node(adata_path=args.adata_path, n_top_genes=args.n_source,cell_type_visualize=True)
trainloader, testloader = human_node_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,
                   n_spot_types=dataset.n_spot_types)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

  Slice slice1: Leiden found 9 local clusters (sizes: [742, 713, 563, 370, 337, 330, 182, 160, 87])
  Slice slice2: Leiden found 9 local clusters (sizes: [781, 631, 485, 398, 362, 298, 168, 129, 107])
  Reference slice slice2 defines the fixed global cell-type space with 9 classes
    Align slice=slice1 local_cluster=0 -> reference/global=0 cosine_similarity=0.9869
    Align slice=slice1 local_cluster=1 -> reference/global=4 cosine_similarity=0.9889
    Align slice=slice1 local_cluster=2 -> reference/global=2 cosine_similarity=0.9770
    Align slice=slice1 local_cluster=3 -> reference/global=3 cosine_similarity=0.9896
    Align slice=slice1 local_cluster=4 -> reference/global=6 cosine_similarity=0.9590
    Align slice=slice1 local_cluster=5 -> reference/global=2 cosine_similarity=0.9931
    Align slice=slice1 local_cluster=6 -> reference/global=8 cosine_similarity=0.9796
    Align slice=slice1 local_cluster=7 -> reference/global=7 cosine_similarity=0.9916
    Align slice=slice1 local_c

### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_human_lymph_node_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 108/108	 Loss 0.900976 (0.975303)
==> Epoch 2/20
Batch 108/108	 Loss 0.724781 (0.875235)
==> Epoch 3/20
Batch 108/108	 Loss 0.625533 (0.857006)
==> Epoch 4/20
Batch 108/108	 Loss 0.363724 (0.821318)
==> Epoch 5/20
Batch 108/108	 Loss 1.910378 (0.803718)
==> Epoch 6/20
Batch 108/108	 Loss 2.309775 (0.796480)
==> Epoch 7/20
Batch 108/108	 Loss 0.402865 (0.772501)
==> Epoch 8/20
Batch 108/108	 Loss 0.413947 (0.742297)
==> Epoch 9/20
Batch 108/108	 Loss 1.057736 (0.751460)
==> Epoch 10/20
Batch 108/108	 Loss 0.522987 (0.744453)
==> Epoch 11/20
Batch 108/108	 Loss 0.409681 (0.708718)
==> Epoch 12/20
Batch 108/108	 Loss 1.129267 (0.690208)
==> Epoch 13/20
Batch 108/108	 Loss 2.173600 (0.681384)
==> Epoch 14/20
Batch 108/108	 Loss 0.359906 (0.683891)
==> Epoch 15/20
Batch 108/108	 Loss 0.294983 (0.679590)
==> Epoch 16/20
Batch 108/108	 Loss 0.912338 (0.662652)
==> Epoch 17/20
Batch 108/108	 Loss 0.362834 (0.667050)
==> Epoch 18/20
Batch 108/108	 Loss 0.795207 (0.661978)
=